In [ ]:
# Pipeline parameters — overridden by job base_parameters when run via DAB.
dbutils.widgets.text("catalog", "actuarial")
dbutils.widgets.text("schema", "dev")
dbutils.widgets.text("volume_name", "raw_files")
dbutils.widgets.text("bronze_write_mode", "overwrite")
dbutils.widgets.text("overwrite_schema", "true")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume_name = dbutils.widgets.get("volume_name")
bronze_write_mode = dbutils.widgets.get("bronze_write_mode")
overwrite_schema = dbutils.widgets.get("overwrite_schema").lower() == "true"
volume_path = f"/Volumes/{catalog}/{schema}/{volume_name}"

print(f"catalog={catalog}  schema={schema}  volume_path={volume_path}")
print(f"bronze_write_mode={bronze_write_mode}  overwrite_schema={overwrite_schema}")

# Gold / Reporting Layer

## Step 6 - Create Gold Tables

| Gold Table | Joins | Purpose |
|---|---|---|
| `gold_claims_summary` | claims + premiums | Frequency, severity & settlement by peril, status, region |
| `gold_loss_ratio_by_risk` | premiums + claims | Loss ratio by insurer, risk band, building type, mitigation |
| `gold_event_loss_summary` | claims + events + premiums | Cat vs non-cat losses per named cyclone event |
| `gold_portfolio_exposure` | premiums + risk zone | Sum insured, policy count & premium rate by risk segment |
| `gold_claims_development` | claims + premiums | Reporting lag, IBNR indicators & reserve by peril & month |


In [ ]:
spark.sql(f"""
-- Claims frequency, severity and settlement progress
-- Joins: silver_claims_bordereau → silver_premium_bordereau on policy_id
CREATE OR REPLACE TABLE {catalog}.{schema}.gold_claims_summary AS
SELECT
  c.peril_type,
  c.claim_status,
  p.region_name,
  p.wind_risk_band,
  p.building_type,
  COUNT(c.claim_id)                                                         AS claim_count,
  SUM(c.incurred_amount)                                                    AS total_incurred,
  SUM(c.paid_to_date)                                                       AS total_paid,
  SUM(c.incurred_amount - c.paid_to_date)                                   AS outstanding_reserve,
  ROUND(AVG(c.incurred_amount), 2)                                          AS avg_claim_severity,
  ROUND(SUM(c.paid_to_date) / NULLIF(SUM(c.incurred_amount), 0) * 100, 2)  AS settlement_pct,
  current_timestamp()                                                       AS gold_ingestion_timestamp
FROM {catalog}.{schema}.silver_claims_bordereau   c
LEFT JOIN {catalog}.{schema}.silver_premium_bordereau p ON c.policy_id = p.policy_id
GROUP BY ALL
""")

In [ ]:
spark.sql(f"""
-- Loss ratio by insurer, region, risk band, building type and mitigation flag
-- Joins: silver_premium_bordereau LEFT JOIN silver_claims_bordereau on policy_id
CREATE OR REPLACE TABLE {catalog}.{schema}.gold_loss_ratio_by_risk AS
SELECT
  p.insurer_name,
  p.region_name,
  p.wind_risk_band,
  p.building_type,
  p.mitigation_flag,
  COUNT(DISTINCT p.policy_id)                                                       AS policy_count,
  SUM(p.annual_premium)                                                             AS total_premium,
  COUNT(c.claim_id)                                                                 AS claim_count,
  COALESCE(SUM(c.incurred_amount), 0)                                               AS total_incurred,
  ROUND(COALESCE(SUM(c.incurred_amount), 0) / NULLIF(SUM(p.annual_premium), 0) * 100, 2) AS loss_ratio_pct,
  current_timestamp()                                                               AS gold_ingestion_timestamp
FROM {catalog}.{schema}.silver_premium_bordereau  p
LEFT JOIN {catalog}.{schema}.silver_claims_bordereau c ON p.policy_id = c.policy_id
GROUP BY ALL
""")

In [ ]:
spark.sql(f"""
-- Cat vs non-cat loss aggregation per named cyclone event
-- Joins: silver_claims_bordereau LEFT JOIN silver_cyclone_events on event_id
--                                LEFT JOIN silver_premium_bordereau on policy_id
CREATE OR REPLACE TABLE {catalog}.{schema}.gold_event_loss_summary AS
SELECT
  CASE WHEN c.event_id IS NOT NULL THEN 'Catastrophe' ELSE 'Non-Catastrophe' END AS claim_category,
  e.event_name,
  e.start_date                                             AS event_start,
  e.end_date                                               AS event_end,
  DATEDIFF(e.end_date, e.start_date)                       AS event_duration_days,
  p.region_name,
  c.peril_type,
  COUNT(c.claim_id)                                        AS claim_count,
  SUM(c.incurred_amount)                                   AS total_incurred,
  ROUND(AVG(c.incurred_amount), 2)                         AS avg_claim_severity,
  MAX(c.incurred_amount)                                   AS max_claim_severity,
  SUM(c.paid_to_date)                                      AS total_paid,
  SUM(c.incurred_amount - c.paid_to_date)                  AS outstanding_reserve,
  current_timestamp()                                      AS gold_ingestion_timestamp
FROM {catalog}.{schema}.silver_claims_bordereau          c
LEFT JOIN {catalog}.{schema}.silver_cyclone_events       e  ON c.event_id  = e.event_id
LEFT JOIN {catalog}.{schema}.silver_premium_bordereau    p  ON c.policy_id = p.policy_id
GROUP BY ALL
""")

In [ ]:
spark.sql(f"""
-- Portfolio exposure: sum insured, policy count and premium rate per risk segment
-- Joins: silver_premium_bordereau INNER JOIN silver_risk_zone_lookup on postcode
CREATE OR REPLACE TABLE {catalog}.{schema}.gold_portfolio_exposure AS
SELECT
  p.insurer_name,
  p.region_name,
  p.wind_risk_band,
  p.building_type,
  p.mitigation_flag,
  COUNT(DISTINCT p.policy_id)                                                        AS policy_count,
  SUM(p.sum_insured)                                                                 AS total_sum_insured,
  SUM(p.annual_premium)                                                              AS total_annual_premium,
  ROUND(AVG(p.sum_insured), 2)                                                       AS avg_sum_insured,
  ROUND(AVG(p.annual_premium), 2)                                                    AS avg_annual_premium,
  ROUND(SUM(p.annual_premium) / NULLIF(SUM(p.sum_insured), 0) * 100, 4)             AS premium_rate_pct,
  current_timestamp()                                                                AS gold_ingestion_timestamp
FROM {catalog}.{schema}.silver_premium_bordereau   p
INNER JOIN {catalog}.{schema}.silver_risk_zone_lookup rz ON p.postcode = rz.postcode
GROUP BY ALL
""")

In [ ]:
spark.sql(f"""
-- Reporting lag, IBNR indicators and outstanding reserve by peril, region and loss month
-- Joins: silver_claims_bordereau LEFT JOIN silver_premium_bordereau on policy_id
CREATE OR REPLACE TABLE {catalog}.{schema}.gold_claims_development AS
SELECT
  c.peril_type,
  p.region_name,
  p.wind_risk_band,
  DATE_TRUNC('month', c.date_of_loss)                                              AS loss_month,
  DATE_TRUNC('month', c.reported_date)                                             AS reported_month,
  COUNT(c.claim_id)                                                                AS claim_count,
  ROUND(AVG(DATEDIFF(c.reported_date, c.date_of_loss)), 1)                         AS avg_reporting_lag_days,
  MAX(DATEDIFF(c.reported_date, c.date_of_loss))                                   AS max_reporting_lag_days,
  SUM(c.incurred_amount)                                                           AS total_incurred,
  SUM(c.paid_to_date)                                                              AS total_paid,
  SUM(c.incurred_amount - c.paid_to_date)                                          AS outstanding_reserve,
  ROUND(SUM(c.paid_to_date) / NULLIF(SUM(c.incurred_amount), 0) * 100, 2)         AS payment_progress_pct,
  current_timestamp()                                                              AS gold_ingestion_timestamp
FROM {catalog}.{schema}.silver_claims_bordereau    c
LEFT JOIN {catalog}.{schema}.silver_premium_bordereau p ON c.policy_id = p.policy_id
GROUP BY ALL
""")